In [ ]:
#@title Copyright 2025 The Earth Engine Community Authors { display-mode: "form" }
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Supervised Classification untuk Benthic Habitat dengan Ground Truth Data

## Part 2: Menggunakan Data Survey Lapangan

Notebook ini adalah **lanjutan dari Part 1** (index.ipynb). Di sini kita akan:

1. **Load shapefile ground truth** dari hasil survey lapangan
2. **Extract training samples** dari imagery yang sudah di-deglint
3. **Supervised classification** menggunakan:
   - Random Forest
   - Support Vector Machine (SVM)
   - Classification and Regression Trees (CART)
4. **Accuracy assessment** dengan confusion matrix
5. **Reclassify** hasil K-Means berdasarkan ground truth

**Prasyarat:**
- Sudah menjalankan Part 1 (deglinting & DII calculation)
- Punya shapefile ground truth dengan kolom yang berisi class/habitat type

**Format Shapefile Ground Truth:**
- Bisa berupa **points** atau **polygons**
- Harus punya kolom yang berisi **nama habitat** (contoh: 'class', 'habitat', 'type')
- Contoh isi kolom: 'coral', 'seagrass', 'sand', 'rock', dll.

## Setup dan Install Libraries

In [ ]:
# Install required packages
!pip install earthengine-api rasterio geopandas scikit-learn matplotlib numpy seaborn -q

In [ ]:
# Import libraries
import ee
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from google.colab import drive
import folium
import os

print("✓ Libraries imported successfully!")

## Authenticate Earth Engine

In [ ]:
# Authenticate and initialize
ee.Authenticate()
ee.Initialize(project='your-project-id')  # Ganti dengan project ID Anda

print("✓ Earth Engine initialized!")

## Mount Google Drive dan Load Data

In [ ]:
# Mount Google Drive
drive.mount('/content/drive')

# Path ke data Anda
drive_path = '/content/drive/MyDrive/sentinel_benthic_data/'
print(f"Data directory: {drive_path}")

## 1. Load Ground Truth Shapefile dari Survey Lapangan

**PENTING:** Sesuaikan nama file dan nama kolom dengan data Anda!

In [ ]:
# ========================================
# EDIT BAGIAN INI SESUAI DATA ANDA
# ========================================

# Nama file shapefile ground truth
ground_truth_file = 'ground_truth.shp'  # Ganti dengan nama file Anda

# Nama kolom yang berisi class/habitat type
class_column = 'habitat'  # Ganti dengan nama kolom Anda (bisa 'class', 'type', 'habitat', dll)

# ========================================

# Load shapefile
gt_path = os.path.join(drive_path, ground_truth_file)
ground_truth = gpd.read_file(gt_path)

print(f"✓ Ground truth loaded: {len(ground_truth)} features")
print(f"\nCRS: {ground_truth.crs}")
print(f"\nGeometry type: {ground_truth.geometry.type.unique()}")
print(f"\nColumns: {list(ground_truth.columns)}")
print(f"\n--- Preview Data ---")
print(ground_truth.head())

# Cek unique classes
if class_column in ground_truth.columns:
    unique_classes = ground_truth[class_column].unique()
    print(f"\n✓ Found {len(unique_classes)} habitat classes:")
    for i, cls in enumerate(unique_classes):
        count = len(ground_truth[ground_truth[class_column] == cls])
        print(f"  {i+1}. {cls}: {count} samples")
else:
    print(f"\n⚠️ WARNING: Column '{class_column}' not found!")
    print(f"Available columns: {list(ground_truth.columns)}")
    print("Please update the 'class_column' variable above.")

### Visualisasi Ground Truth Data

In [ ]:
# Plot ground truth distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Map view
ground_truth.plot(column=class_column, categorical=True, legend=True, ax=ax1, 
                  cmap='Set3', edgecolor='black', linewidth=0.5)
ax1.set_title('Ground Truth Spatial Distribution', fontsize=14, fontweight='bold')
ax1.set_xlabel('Longitude')
ax1.set_ylabel('Latitude')

# Class count
class_counts = ground_truth[class_column].value_counts()
class_counts.plot(kind='bar', ax=ax2, color='steelblue', edgecolor='black')
ax2.set_title('Number of Samples per Class', fontsize=14, fontweight='bold')
ax2.set_xlabel('Habitat Class')
ax2.set_ylabel('Count')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print(f"\nTotal ground truth samples: {len(ground_truth)}")
print(f"Classes are well-distributed: {'✓ Yes' if class_counts.min() >= 10 else '⚠️ No - some classes have < 10 samples'}")

## 2. Load Sentinel-2 Image dan Apply Deglinting + DII

**Ini sama dengan Part 1**, tapi kita gabungkan dalam satu proses.

In [ ]:
# Load AOI dari ground truth extent
bounds = ground_truth.total_bounds
aoi = ee.Geometry.Rectangle([bounds[0], bounds[1], bounds[2], bounds[3]])

print(f"AOI bounds: {bounds}")

# Date range - EDIT SESUAI KEBUTUHAN
start_date = '2024-01-01'
end_date = '2024-12-31'

# Load Sentinel-2
s2_collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                .filterBounds(aoi)
                .filterDate(start_date, end_date)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))
                .sort('CLOUDY_PIXEL_PERCENTAGE'))

# Get least cloudy image
s2_image = s2_collection.first()
s2_image = s2_image.select(['B2', 'B3', 'B4', 'B8'])

image_date = ee.Date(s2_image.get('system:time_start')).format('YYYY-MM-dd').getInfo()
cloud_cover = s2_image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()

print(f"\n✓ Sentinel-2 image loaded")
print(f"  Date: {image_date}")
print(f"  Cloud cover: {cloud_cover:.2f}%")
print(f"  Available images in range: {s2_collection.size().getInfo()}")

In [ ]:
# Function: Deglinting (dari Part 1)
def hedley_deglint(image, nir_band='B8', min_nir_percentile=1):
    visible_bands = ['B2', 'B3', 'B4']
    nir = image.select(nir_band)
    
    min_nir = nir.reduceRegion(
        reducer=ee.Reducer.percentile([min_nir_percentile]),
        geometry=aoi,
        scale=10,
        maxPixels=1e9
    ).get(nir_band)
    
    nir_diff = nir.subtract(ee.Number(min_nir))
    deglinted_bands = []
    
    for band in visible_bands:
        sample = image.select([band, nir_band]).sample(
            region=aoi, scale=10, numPixels=500, seed=42
        )
        regression = sample.reduceColumns(
            reducer=ee.Reducer.linearFit(),
            selectors=[nir_band, band]
        )
        slope = ee.Number(regression.get('scale'))
        deglinted = image.select(band).subtract(nir_diff.multiply(slope))
        deglinted_bands.append(deglinted.rename(band + '_deglint'))
    
    return ee.Image.cat(deglinted_bands).addBands(nir)

# Function: Calculate DII (dari Part 1)
def calculate_dii_indices(image):
    b2 = image.select('B2_deglint').add(0.0001)
    b3 = image.select('B3_deglint').add(0.0001)
    b4 = image.select('B4_deglint').add(0.0001)
    b8 = image.select('B8')
    
    dii_b2_b3 = b2.log().divide(b3.log()).rename('DII_B2_B3')
    dii_b3_b4 = b3.log().divide(b4.log()).rename('DII_B3_B4')
    ndwi = b3.subtract(b8).divide(b3.add(b8)).rename('NDWI')
    ratio_b2_b3 = b2.divide(b3).rename('Ratio_B2_B3')
    ratio_b3_b4 = b3.divide(b4).rename('Ratio_B3_B4')
    nd_b2_b3 = b2.subtract(b3).divide(b2.add(b3)).rename('ND_B2_B3')
    nd_b3_b4 = b3.subtract(b4).divide(b3.add(b4)).rename('ND_B3_B4')
    
    return ee.Image.cat([
        image, dii_b2_b3, dii_b3_b4, ndwi,
        ratio_b2_b3, ratio_b3_b4, nd_b2_b3, nd_b3_b4
    ])

# Apply processing
print("Processing image...")
s2_deglinted = hedley_deglint(s2_image)
s2_with_dii = calculate_dii_indices(s2_deglinted)

print("✓ Deglinting completed")
print("✓ DII indices calculated")
print(f"\nAvailable bands: {s2_with_dii.bandNames().getInfo()}")

## 3. Extract Training Data dari Ground Truth

Kita akan extract nilai pixel dari imagery untuk setiap titik ground truth.

In [ ]:
# Pilih bands untuk classification
classification_bands = [
    'B2_deglint', 'B3_deglint', 'B4_deglint', 'B8',
    'DII_B2_B3', 'DII_B3_B4', 'NDWI',
    'Ratio_B2_B3', 'Ratio_B3_B4'
]

classification_image = s2_with_dii.select(classification_bands)

# Convert ground truth to Earth Engine FeatureCollection
def gdf_to_ee_features(gdf, class_col):
    """Convert GeoDataFrame to Earth Engine FeatureCollection"""
    features = []
    for idx, row in gdf.iterrows():
        geom = row.geometry
        if geom.geom_type == 'Point':
            ee_geom = ee.Geometry.Point([geom.x, geom.y])
        elif geom.geom_type == 'Polygon':
            coords = list(geom.exterior.coords)
            ee_geom = ee.Geometry.Polygon(coords)
        else:
            continue
        
        feature = ee.Feature(ee_geom, {'class': str(row[class_col])})
        features.append(feature)
    
    return ee.FeatureCollection(features)

print("Converting ground truth to Earth Engine format...")
ground_truth_ee = gdf_to_ee_features(ground_truth, class_column)

print(f"✓ Converted {ground_truth_ee.size().getInfo()} ground truth features")

# Sample the image at ground truth locations
print("\nExtracting training data from imagery...")
training_data = classification_image.sampleRegions(
    collection=ground_truth_ee,
    properties=['class'],
    scale=10,
    tileScale=4
)

n_samples = training_data.size().getInfo()
print(f"✓ Extracted {n_samples} training samples")

# Check if we have enough samples
if n_samples < 50:
    print("\n⚠️ WARNING: Less than 50 training samples!")
    print("Consider adding more ground truth points or using polygon sampling.")
else:
    print("✓ Sufficient training samples for classification")

## 4. Supervised Classification

Kita akan train 3 classifier berbeda dan bandingkan hasilnya:
1. **Random Forest** (biasanya paling akurat)
2. **Support Vector Machine (SVM)**
3. **CART (Decision Tree)**

In [ ]:
# Split training data untuk validation
training_sample = training_data.randomColumn('random', seed=42)
training_set = training_sample.filter(ee.Filter.lt('random', 0.7))  # 70% training
validation_set = training_sample.filter(ee.Filter.gte('random', 0.7))  # 30% validation

print(f"Training samples: {training_set.size().getInfo()}")
print(f"Validation samples: {validation_set.size().getInfo()}")

# ============================================
# 1. RANDOM FOREST CLASSIFIER
# ============================================
print("\n" + "="*50)
print("TRAINING RANDOM FOREST CLASSIFIER")
print("="*50)

rf_classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=100,
    variablesPerSplit=3,
    minLeafPopulation=1,
    bagFraction=0.5,
    seed=42
).train(
    features=training_set,
    classProperty='class',
    inputProperties=classification_bands
)

rf_classified = classification_image.classify(rf_classifier)
print("✓ Random Forest training completed")

# ============================================
# 2. SUPPORT VECTOR MACHINE (SVM)
# ============================================
print("\n" + "="*50)
print("TRAINING SVM CLASSIFIER")
print("="*50)

svm_classifier = ee.Classifier.libsvm(
    kernelType='RBF',
    gamma=0.5,
    cost=10
).train(
    features=training_set,
    classProperty='class',
    inputProperties=classification_bands
)

svm_classified = classification_image.classify(svm_classifier)
print("✓ SVM training completed")

# ============================================
# 3. CART (Decision Tree)
# ============================================
print("\n" + "="*50)
print("TRAINING CART CLASSIFIER")
print("="*50)

cart_classifier = ee.Classifier.smileCart(
    maxNodes=50
).train(
    features=training_set,
    classProperty='class',
    inputProperties=classification_bands
)

cart_classified = classification_image.classify(cart_classifier)
print("✓ CART training completed")

print("\n" + "="*50)
print("✓ ALL CLASSIFIERS TRAINED SUCCESSFULLY!")
print("="*50)

## 5. Accuracy Assessment

Evaluate accuracy menggunakan validation set

In [ ]:
# Validate Random Forest
rf_validated = validation_set.classify(rf_classifier)
rf_error_matrix = rf_validated.errorMatrix('class', 'classification')
rf_accuracy = rf_error_matrix.accuracy().getInfo()
rf_kappa = rf_error_matrix.kappa().getInfo()

# Validate SVM
svm_validated = validation_set.classify(svm_classifier)
svm_error_matrix = svm_validated.errorMatrix('class', 'classification')
svm_accuracy = svm_error_matrix.accuracy().getInfo()
svm_kappa = svm_error_matrix.kappa().getInfo()

# Validate CART
cart_validated = validation_set.classify(cart_classifier)
cart_error_matrix = cart_validated.errorMatrix('class', 'classification')
cart_accuracy = cart_error_matrix.accuracy().getInfo()
cart_kappa = cart_error_matrix.kappa().getInfo()

# Display results
print("\n" + "="*60)
print("ACCURACY ASSESSMENT RESULTS")
print("="*60)

results_df = pd.DataFrame({
    'Classifier': ['Random Forest', 'SVM', 'CART'],
    'Overall Accuracy': [rf_accuracy, svm_accuracy, cart_accuracy],
    'Kappa Coefficient': [rf_kappa, svm_kappa, cart_kappa]
})

print(results_df.to_string(index=False))
print("\n" + "="*60)

# Determine best classifier
best_classifier_idx = results_df['Overall Accuracy'].idxmax()
best_classifier_name = results_df.loc[best_classifier_idx, 'Classifier']
best_accuracy = results_df.loc[best_classifier_idx, 'Overall Accuracy']

print(f"\n🏆 BEST CLASSIFIER: {best_classifier_name}")
print(f"   Overall Accuracy: {best_accuracy*100:.2f}%")
print(f"   Kappa: {results_df.loc[best_classifier_idx, 'Kappa Coefficient']:.3f}")

# Bar plot
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results_df))
width = 0.35

ax.bar(x - width/2, results_df['Overall Accuracy'], width, label='Overall Accuracy', color='steelblue')
ax.bar(x + width/2, results_df['Kappa Coefficient'], width, label='Kappa', color='coral')

ax.set_ylabel('Score', fontweight='bold')
ax.set_title('Classifier Performance Comparison', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(results_df['Classifier'])
ax.legend()
ax.set_ylim([0, 1.1])
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

### Confusion Matrix untuk Best Classifier

In [ ]:
# Get confusion matrix for best classifier (usually Random Forest)
if best_classifier_name == 'Random Forest':
    error_matrix = rf_error_matrix
elif best_classifier_name == 'SVM':
    error_matrix = svm_error_matrix
else:
    error_matrix = cart_error_matrix

# Get confusion matrix as array
confusion = np.array(error_matrix.getInfo())

# Get class names from ground truth
class_names = sorted(ground_truth[class_column].unique())

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title(f'Confusion Matrix - {best_classifier_name}', fontsize=14, fontweight='bold')
plt.ylabel('True Class', fontweight='bold')
plt.xlabel('Predicted Class', fontweight='bold')
plt.tight_layout()
plt.show()

# Calculate per-class accuracy
print("\nPer-class Producer's Accuracy (Recall):")
print("="*50)
producers_accuracy = error_matrix.producersAccuracy().getInfo()
for i, cls in enumerate(class_names):
    print(f"{cls:20s}: {producers_accuracy[i]*100:6.2f}%")

print("\nPer-class User's Accuracy (Precision):")
print("="*50)
consumers_accuracy = error_matrix.consumersAccuracy().getInfo()
for i, cls in enumerate(class_names):
    print(f"{cls:20s}: {consumers_accuracy[i]*100:6.2f}%")

## 6. Visualisasi Hasil Klasifikasi

Bandingkan ketiga classifier secara visual

In [ ]:
# Helper function untuk add EE layer ke folium
def add_ee_layer(self, ee_image_object, vis_params, name, show=True, opacity=1):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Google Earth Engine',
        name=name,
        show=show,
        opacity=opacity,
        overlay=True,
        control=True
    ).add_to(self)

folium.Map.add_ee_layer = add_ee_layer

# Create map
center = [ground_truth.geometry.centroid.y.mean(), ground_truth.geometry.centroid.x.mean()]
m = folium.Map(location=center, zoom_start=13)

# RGB visualization
vis_rgb = {
    'bands': ['B4_deglint', 'B3_deglint', 'B2_deglint'],
    'min': 0,
    'max': 3000,
    'gamma': 1.4
}

# Classification visualization
# Create color palette based on number of classes
n_classes = len(class_names)
colors = ['#001a4d', '#00ff00', '#ffff00', '#ff7f00', '#ff0000', '#ff00ff', '#00ffff', '#8b4513']
class_colors = colors[:n_classes]

vis_classified = {
    'min': 0,
    'max': n_classes - 1,
    'palette': class_colors
}

# Add layers
m.add_ee_layer(s2_deglinted, vis_rgb, 'Deglinted RGB', True, 0.7)
m.add_ee_layer(rf_classified, vis_classified, f'Random Forest (Acc: {rf_accuracy*100:.1f}%)', False, 0.6)
m.add_ee_layer(svm_classified, vis_classified, f'SVM (Acc: {svm_accuracy*100:.1f}%)', False, 0.6)
m.add_ee_layer(cart_classified, vis_classified, f'CART (Acc: {cart_accuracy*100:.1f}%)', False, 0.6)

# Add ground truth points
for idx, row in ground_truth.iterrows():
    if row.geometry.geom_type == 'Point':
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=3,
            popup=f"Class: {row[class_column]}",
            color='red',
            fill=True,
            fillColor='red',
            fillOpacity=0.8
        ).add_to(m)

# Add layer control
m.add_child(folium.LayerControl())

# Display map
display(m)

print("\n🗺️ Map Legend:")
for i, cls in enumerate(class_names):
    print(f"  {cls}: Color {i}")
print("\n🔴 Red dots: Ground truth locations")

## 7. Export Hasil Klasifikasi

Export best classifier results ke Google Drive

In [ ]:
# Select best classifier result
if best_classifier_name == 'Random Forest':
    best_classified = rf_classified
elif best_classifier_name == 'SVM':
    best_classified = svm_classified
else:
    best_classified = cart_classified

# Export classified image
export_task = ee.batch.Export.image.toDrive(
    image=best_classified,
    description=f'benthic_supervised_{best_classifier_name.replace(" ", "_")}',
    folder='sentinel_benthic_results',
    region=aoi,
    scale=10,
    crs='EPSG:4326',
    maxPixels=1e9
)

export_task.start()

print("="*60)
print("EXPORT STARTED")
print("="*60)
print(f"Classifier: {best_classifier_name}")
print(f"Accuracy: {best_accuracy*100:.2f}%")
print(f"Task ID: {export_task.id}")
print(f"\nCheck your Earth Engine Tasks tab to monitor progress.")
print(f"Result will be saved to Google Drive: sentinel_benthic_results/")

# Save accuracy report
accuracy_report = f"""
BENTHIC HABITAT CLASSIFICATION REPORT
{'='*60}

Date: {image_date}
Cloud Cover: {cloud_cover:.2f}%

GROUND TRUTH DATA:
- Total samples: {len(ground_truth)}
- Number of classes: {len(class_names)}
- Classes: {', '.join(class_names)}

CLASSIFICATION RESULTS:
{results_df.to_string(index=False)}

BEST CLASSIFIER: {best_classifier_name}
- Overall Accuracy: {best_accuracy*100:.2f}%
- Kappa Coefficient: {results_df.loc[best_classifier_idx, 'Kappa Coefficient']:.3f}

CONFUSION MATRIX:
{confusion}
"""

print("\n" + accuracy_report)

# Save to file (optional)
report_path = os.path.join(drive_path, 'classification_report.txt')
with open(report_path, 'w') as f:
    f.write(accuracy_report)

print(f"\n✓ Report saved to: {report_path}")

## 8. Analisis Area per Habitat Class

In [ ]:
# Calculate area for each class
pixel_area = ee.Image.pixelArea()

class_areas = best_classified.addBands(pixel_area).reduceRegion(
    reducer=ee.Reducer.sum().group(
        groupField=0,
        groupName='class'
    ),
    geometry=aoi,
    scale=10,
    maxPixels=1e9
)

# Extract results
area_info = class_areas.getInfo()

print("\n" + "="*60)
print("HABITAT AREA DISTRIBUTION")
print("="*60)

area_data = []
total_area = 0

for item in area_info['groups']:
    class_id = int(item['class'])
    if class_id < len(class_names):
        class_name = class_names[class_id]
        area_m2 = item['sum']
        area_ha = area_m2 / 10000
        total_area += area_ha
        area_data.append({'Class': class_name, 'Area (ha)': area_ha, 'Area (m²)': area_m2})
        print(f"{class_name:20s}: {area_ha:10.2f} ha ({area_m2:,.0f} m²)")

print("="*60)
print(f"{'TOTAL':20s}: {total_area:10.2f} ha")

# Create visualizations
area_df = pd.DataFrame(area_data)
area_df['Percentage'] = (area_df['Area (ha)'] / total_area * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Pie chart
ax1.pie(area_df['Area (ha)'], labels=area_df['Class'], autopct='%1.1f%%', 
        startangle=90, colors=class_colors)
ax1.set_title('Habitat Distribution by Area', fontsize=14, fontweight='bold')

# Bar chart
ax2.barh(area_df['Class'], area_df['Area (ha)'], color=class_colors, edgecolor='black')
ax2.set_xlabel('Area (hectares)', fontweight='bold')
ax2.set_title('Habitat Area Comparison', fontsize=14, fontweight='bold')
ax2.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

# Save area statistics
area_csv_path = os.path.join(drive_path, 'habitat_area_statistics.csv')
area_df.to_csv(area_csv_path, index=False)
print(f"\n✓ Area statistics saved to: {area_csv_path}")

## 📊 Summary

### Yang Sudah Kita Lakukan:

✅ **Load ground truth** dari hasil survey lapangan  
✅ **Extract training data** dari Sentinel-2 imagery  
✅ **Train 3 classifier** (Random Forest, SVM, CART)  
✅ **Accuracy assessment** dengan confusion matrix  
✅ **Visualisasi hasil** klasifikasi  
✅ **Export** hasil terbaik ke Google Drive  
✅ **Analisis area** per habitat class  

### Output Files:

1. **Classified image GeoTIFF** - di Google Drive folder `sentinel_benthic_results/`
2. **Classification report** - `classification_report.txt`
3. **Area statistics** - `habitat_area_statistics.csv`

### Next Steps:

- **Validasi hasil** dengan survey lapangan tambahan
- **Temporal analysis** - jalankan untuk multiple dates
- **Post-processing** - filter dengan morphological operations
- **Integration dengan GIS** - import ke QGIS/ArcGIS untuk analisis lanjutan

### Tips untuk Meningkatkan Akurasi:

1. **Tambah ground truth samples** - minimal 50 per class
2. **Balance class distribution** - usahakan jumlah sample tiap class seimbang
3. **Tune classifier parameters** - adjust numberOfTrees, gamma, dll
4. **Feature selection** - test dengan band combinations berbeda
5. **Multi-date analysis** - composite dari multiple images

---

**Selamat! Anda sudah berhasil melakukan supervised classification untuk benthic habitat mapping! 🎉**